## Example 1: Manipulate Data Lake paths with AzureDLPath

In [ ]:
# Example 1: Manipulate Data Lake paths with AzureDLPath
from azure.datalake.store.core import AzureDLPath

path = AzureDLPath("/data/logs/2024/events.csv")
assert path.name == "events.csv"
assert path.suffix == ".csv"
assert str(path.parent) == "/data/logs/2024"
# trim() drops the leading slash; the store addresses paths relative to its root
assert str(path.trim()) == "data/logs/2024/events.csv"
assert str(AzureDLPath("data/logs").trim()) == "data/logs"
# globless_prefix is the directory part in front of the first wildcard
assert str(AzureDLPath("/data/logs/*/events.csv").globless_prefix) == "/data/logs"
print(f"name={path.name} parent={path.parent} trimmed={path.trim()}")

## Example 2: Check transfer and open() defaults in the real signatures

In [ ]:
# Example 2: Check transfer and open() defaults in the real signatures
import inspect

from azure.datalake.store.core import AzureDLFileSystem
from azure.datalake.store.multithread import ADLDownloader, ADLUploader

uploader = inspect.signature(ADLUploader).parameters
downloader = inspect.signature(ADLDownloader).parameters
open_params = inspect.signature(AzureDLFileSystem.open).parameters

# nthreads=None means "use the number of CPU cores" in both directions
assert uploader["nthreads"].default is None
assert downloader["nthreads"].default is None
for params in (uploader, downloader):
    assert {"chunksize", "buffersize", "blocksize", "overwrite", "progress_callback"} <= set(params)

# open() takes a path, a mode, a block size and a record delimiter, and reads binary by default
assert list(open_params) == ["self", "path", "mode", "blocksize", "delimiter"], list(open_params)
assert open_params["mode"].default == "rb"
print("ADLUploader parameters:", list(uploader))
print("open() blocksize default:", open_params["blocksize"].default)

## Example 3: List, inspect and read files through a canned HTTP adapter

In [ ]:
# Example 3: List, inspect and read files through a canned HTTP adapter
# AzureDLFileSystem talks WebHDFS over a requests session. CannedAdapter answers each request
# from the in-memory store below, so no request leaves the process; FakeCredential hands out a
# fixed token.
import http
import io
import json
import time
from urllib.parse import parse_qs, urlsplit

import requests
from azure.core.credentials import AccessToken
from azure.datalake.store.core import AzureDLFileSystem


class CannedAdapter(requests.adapters.BaseAdapter):
    def __init__(self, handler):
        super().__init__()
        self.handler = handler  # request -> (status code, JSON payload or raw bytes, extra headers)
        self.requests = []

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload, headers = self.handler(request)
        body = payload if isinstance(payload, bytes) else json.dumps(payload).encode()
        response = requests.Response()
        response.status_code = status_code
        response.reason = http.HTTPStatus(status_code).phrase
        response.headers = requests.structures.CaseInsensitiveDict(
            {"Content-Type": "application/json", "Content-Length": str(len(body)), **headers})
        response.raw = io.BytesIO(body)
        response.url = request.url
        response.request = request
        return response

    def close(self):
        pass


class FakeCredential:
    def get_token(self, *scopes, **kwargs):
        return AccessToken("fake-token", int(time.time()) + 3600)


files = {"data/events.csv": b"id,value\n1,10\n2,20\n"}  # path -> contents
dirs = {"data", "data/archive"}
OCTETS = {"Content-Type": "application/octet-stream"}


def file_status(name, kind, length):
    return {"pathSuffix": name, "type": kind, "length": length, "blockSize": 268435456,
            "modificationTime": 1700000000000, "permission": "770", "owner": "owner", "group": "group"}


def webhdfs(request):
    url = urlsplit(request.url)
    query = parse_qs(url.query)
    op, path = query["OP"][0], url.path.split("/webhdfs/v1", 1)[1].strip("/")
    if path.startswith("secure"):
        return 403, {"RemoteException": {"exception": "AccessControlException", "message": path}}, {}
    if op == "LISTSTATUS" and path in dirs:
        children = sorted({p[len(path) + 1:].split("/")[0] for p in [*files, *dirs] if p.startswith(path + "/")})
        statuses = [file_status(c, "FILE", len(files[f"{path}/{c}"])) if f"{path}/{c}" in files
                    else file_status(c, "DIRECTORY", 0) for c in children]
        return 200, {"FileStatuses": {"FileStatus": statuses, "continuationToken": ""}}, {}
    if op == "GETFILESTATUS" and path in files:
        return 200, {"FileStatus": file_status("", "FILE", len(files[path]))}, {}
    if op == "OPEN" and path in files:
        offset = int(query.get("offset", ["0"])[0])
        length = int(query.get("length", [str(len(files[path]))])[0])
        return 200, files[path][offset:offset + length], OCTETS
    if op == "CREATE":
        files[path] = request.body or b""
        return 201, b"", OCTETS
    if op == "APPEND" and path in files:
        files[path] += request.body or b""
        return 200, b"", OCTETS
    return 404, {"RemoteException": {"exception": "FileNotFoundException", "message": path}}, {}


adapter = CannedAdapter(webhdfs)
adl = AzureDLFileSystem(token_credential=FakeCredential(), store_name="examplestore")
adl.azure.session.mount(adl.azure.url, adapter)

assert adl.ls("/data") == ["data/archive", "data/events.csv"], adl.ls("/data")
kinds = {entry["name"]: entry["type"] for entry in adl.ls("/data", detail=True)}
assert kinds == {"data/archive": "DIRECTORY", "data/events.csv": "FILE"}, kinds
assert adl.info("/data/events.csv")["length"] == len(files["data/events.csv"])
assert adl.cat("/data/events.csv") == b"id,value\n1,10\n2,20\n"
with adl.open("/data/events.csv", "rb") as f:
    assert f.read().splitlines()[1:] == [b"1,10", b"2,20"]
assert all(r.headers["Authorization"] == "Bearer fake-token" for r in adapter.requests)
print(len(adapter.requests), "WebHDFS requests:", sorted({parse_qs(urlsplit(r.url).query)["OP"][0] for r in adapter.requests}))

## Example 4: HTTP errors map to Python exceptions

In [ ]:
# Example 4: HTTP errors map to Python exceptions
# Uses `adl` from Example 3: a 404 becomes FileNotFoundError and a 403 becomes PermissionError.
from azure.datalake.store import exceptions

try:
    adl.info("/data/missing.csv")
except FileNotFoundError as err:
    print("missing file ->", type(err).__name__, err)
else:
    raise AssertionError("info() on a missing path did not raise FileNotFoundError")

try:
    adl.ls("/secure")
except PermissionError as err:
    print("forbidden directory ->", type(err).__name__, err)
else:
    raise AssertionError("ls() on a forbidden path did not raise PermissionError")

assert adl.exists("/data/events.csv")
assert not adl.exists("/data/missing.csv")

# The SDK's own exceptions are OSErrors, so generic I/O handlers catch them too
for cls in (exceptions.DatalakeRESTException, exceptions.DatalakeBadOffsetException,
            exceptions.DatalakeIncompleteTransferException):
    assert issubclass(cls, OSError) and issubclass(cls, Exception), cls

## Example 5: Write a file and read it back

In [ ]:
# Example 5: Write a file and read it back
# Uses `adl` and `files` from Example 3: writes become WebHDFS CREATE and APPEND requests.
with adl.open("/data/report.txt", "wb") as f:
    f.write(b"rows: 2\n")
    f.write(b"total: 30\n")

assert files["data/report.txt"] == b"rows: 2\ntotal: 30\n"
assert adl.cat("/data/report.txt") == b"rows: 2\ntotal: 30\n"
assert adl.info("/data/report.txt")["length"] == len(b"rows: 2\ntotal: 30\n")
assert "data/report.txt" in adl.ls("/data")
print("written:", adl.cat("/data/report.txt"))